# Alternative Win-Probability Model -- Causal Transformer

**Secondary, non-default** entry point. Trains and evaluates the ported hrishav-derived causal Transformer + embedding model (`../src/transformer_model.py`) on project_gagan's data.

hrishav's own bootstrap-CI analysis found this architecture does not beat a simple calibrated Logistic Regression at a statistically significant level -- see `../docs/known_limitations.md`. It is kept here for comparison against `run_all.ipynb`'s default path, not as the recommended model.

**Not comparable to hrishav's own headline numbers** -- this is trained on project_gagan's `ipl_data.xlsx`-derived data (different date range, different season boundaries, different game-state feature computation), not hrishav's original Cricsheet JSON.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')   # resolve src/ from notebooks/

from src.data import load_ball_by_ball
from src.alt_transformer_data import (
    build_player_registry, build_embedding_lookup, build_innings_sequences,
)
from src.alt_transformer_train import train_alt_transformer

DATA_XLSX = '../data/raw/ipl_data.xlsx'

TRAIN_YEARS = set(range(2008, 2019))   # 2008-2018
VAL_YEARS = {2019, 2020}
TEST_YEARS = set(range(2021, 2026))    # 2021-2025, matches run_all.ipynb's internal test window

EPOCHS = 8   # reduce for a faster smoke test

## 2. Load Data & Build Player Embeddings

Fixed random-init embedding table (not GNN-pretrained -- see `../src/transformer_model.py`'s `PlayerEmbedTable` docstring for why).

In [2]:
print('Loading data...')
df = load_ball_by_ball(DATA_XLSX)

registry = build_player_registry(df)
embed_lookup = build_embedding_lookup(registry)
print(f'{len(registry)} unique players')

Loading data...


766 unique players


## 3. Build Innings Sequences (train/val/test)

In [3]:
train_seqs = build_innings_sequences(df, embed_lookup, TRAIN_YEARS)
val_seqs = build_innings_sequences(df, embed_lookup, VAL_YEARS)
test_seqs = build_innings_sequences(df, embed_lookup, TEST_YEARS)

print(f'train={len(train_seqs)}  val={len(val_seqs)}  test={len(test_seqs)} innings')

train=1372  val=226  test=694 innings


## 4. Train the Transformer (win-only)

Takes roughly 1.5-2 minutes for 8 epochs on CPU.

In [4]:
result = train_alt_transformer(train_seqs, val_seqs, test_seqs, epochs=EPOCHS)

print('RESULTS (win probability at the final ball of each innings)')
print(f"Best val Brier: {result['best_val_brier']:.4f}")
if 'test_brier' in result:
    print(f"Test Brier: {result['test_brier']:.4f}  AUC: {result['test_auc']:.4f}  n={result['test_n']}")

RESULTS (win probability at the final ball of each innings)
Best val Brier: 0.1248
Test Brier: 0.1369  AUC: 0.8950  n=694


## Summary

For comparison, `run_all.ipynb`'s default calibrated LogReg/GBT models (team-level features) are the recommended win-probability path -- see `../README.md` and `../docs/known_limitations.md`.